# Stage Two: Database Construction and Customer Value Calculation

This notebook is a record of the code for Stage Two. What each result showed, and the reasoning behind each decision, are set out in `md/02_schema_build_writeup.md`.

---

## 1. Creating the database and building the tables

This opens a DuckDB connection and runs `01_schema.sql`, which creates the four tables, then lists the tables to confirm they exist.

DuckDB runs inside the program without needing a server, which keeps the project easy to run on any machine while still being full SQL.

In [1]:
import duckdb

con = duckdb.connect("../telco.duckdb")

with open("../sql/01_schema.sql") as f:
    con.execute(f.read())

# Confirm the four tables exist
con.execute("SELECT table_name FROM information_schema.tables ORDER BY 1").df()

,table_name
0,billing
1,customer
2,customer_scores
3,geography
4,subscription
5,vw_customer_clv
6,vw_customer_features
7,vw_customer_value_segments


## 2. The structure of each table

The four cells below print the column definitions of each table, so that the data types, primary keys and rules about blank values can be compared against the design.

The particular thing being checked is that `churn_reason`, `churn_score` and `cltv` are the only fields allowed to be blank. Every other field has to be filled in, so a blank appearing in one of them would point to a fault in the load rather than a gap in the data.

In [2]:
con.execute("DESCRIBE customer").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,zip_code,INTEGER,NO,NaN,None,None
2,gender,VARCHAR,NO,NaN,None,None
3,senior_citizen,VARCHAR,NO,NaN,None,None
4,partner,VARCHAR,NO,NaN,None,None
5,dependents,VARCHAR,NO,NaN,None,None
6,tenure_months,INTEGER,NO,NaN,None,None
7,churn_label,VARCHAR,NO,NaN,None,None
8,churn_value,INTEGER,NO,NaN,None,None
9,churn_reason,VARCHAR,YES,NaN,None,None


In [3]:
con.execute("DESCRIBE geography").df()

,column_name,column_type,null,key,default,extra
0,zip_code,INTEGER,NO,PRI,None,None
1,city,VARCHAR,NO,NaN,None,None
2,latitude,DOUBLE,NO,NaN,None,None
3,longitude,DOUBLE,NO,NaN,None,None


In [4]:
con.execute("DESCRIBE billing").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,paperless_billing,VARCHAR,NO,NaN,None,None
2,payment_method,VARCHAR,NO,NaN,None,None
3,monthly_charges,"DECIMAL(10,2)",NO,NaN,None,None
4,total_charges,"DECIMAL(10,2)",NO,NaN,None,None


In [5]:
con.execute("DESCRIBE subscription").df()

,column_name,column_type,null,key,default,extra
0,customer_id,VARCHAR,NO,PRI,None,None
1,phone_service,VARCHAR,NO,NaN,None,None
2,multiple_lines,VARCHAR,NO,NaN,None,None
3,internet_service,VARCHAR,NO,NaN,None,None
4,online_security,VARCHAR,NO,NaN,None,None
5,online_backup,VARCHAR,NO,NaN,None,None
6,device_protection,VARCHAR,NO,NaN,None,None
7,tech_support,VARCHAR,NO,NaN,None,None
8,streaming_tv,VARCHAR,NO,NaN,None,None
9,streaming_movies,VARCHAR,NO,NaN,None,None


## 3. Fetching the source file and making it visible to the database

This downloads the source workbook through `kagglehub`, reads it with pandas, and registers it with the DuckDB connection under the name `raw`. The record count is then confirmed before going further.

Registering the file makes the spreadsheet readable as though it were a table, without copying it into the database. This is where the boundary in the project sits: pandas fetches the file and does nothing else. Every rename, type change and correction happens in SQL in the next step.

In [6]:
import pandas as pd, kagglehub

path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")
raw = pd.read_excel(f"{path}/Telco_customer_churn.xlsx")

con.register("raw", raw)

c:\Users\IC Clearwater\OneDrive\Documents\GitHub\Customer_Lifetime_Value_Churn_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
con.execute("SELECT COUNT(*) FROM raw").df()

,count_star()
0,7043


## 4. Loading the four tables

This runs `02_load.sql`, which fills each table in the required order. `geography` goes first because `customer` holds a reference pointing to it. The row count of each table is confirmed afterwards.

Two changes are made during the load, both deliberate. Repeated addresses are removed from `geography`, since the spreadsheet holds 7,043 rows but only 1,652 postal codes, each repeating once for every customer living there. And `total_charges` is converted from text to a number, with the eleven blank entries found at Stage One set to `0.00` rather than dropped.

In [8]:
with open("../sql/02_load.sql") as f:
    con.execute(f.read())

con.execute("SELECT COUNT(*) FROM geography").df()

,count_star()
0,1652


In [9]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM customer").df()

,count_star()
0,7043


In [10]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM subscription").df()

,count_star()
0,7043


In [11]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("SELECT COUNT(*) FROM billing").df()

,count_star()
0,7043


In [12]:
for f in ["../sql/01_schema.sql", "../sql/02_load.sql"]:
    con.execute(open(f).read())

con.execute("""
    SELECT 'geography' AS t, COUNT(*) FROM geography
    UNION ALL SELECT 'customer', COUNT(*) FROM customer
    UNION ALL SELECT 'subscription', COUNT(*) FROM subscription
    UNION ALL SELECT 'billing', COUNT(*) FROM billing
""").df()

,t,count_star()
0,geography,1652
1,customer,7043
2,subscription,7043
3,billing,7043


## 5. Checking the load

Row counts on their own do not show that the data loaded correctly, so this runs a set of checks against values already known from Stage One.

The checks sit in `sql/02b_validate.sql` rather than being written inline, which keeps the checking visible alongside the rest of the SQL. Each one returns a figure whose expected value is already known, so anything else points to a fault in the load rather than something found in the data.

In [13]:
con.execute(open("../sql/02b_validate.sql").read()).df()

,check,value
0,churned customers,1869
1,churn_reason populated,1869
2,zero total_charges,11
3,zero tenure,11
4,full four-table join,7043


## 6. Building the measures

This runs `03_features.sql`, which creates `vw_customer_features`, then groups customers by tenure band and shows the departure rate in each.

Seven measures are worked out in that file: tenure band, whether the customer holds internet, whether they hold a telephone line, a count of the optional services held, a ranking of how easily the contract can be ended, whether payment is automatic, and a monthly spend band. It is built as a view rather than a stored table. At 7,043 rows the difference in speed is too small to matter, and a view keeps the working of each measure readable instead of hidden inside stored data.

The departure rate by band is shown in order to check that the band boundaries separate customers who behave differently.

In [14]:
con.execute(open("../sql/03_features.sql").read())
con.execute("SELECT tenure_band, COUNT(*), AVG(churn_value) FROM vw_customer_features GROUP BY 1 ORDER BY 1").df()

,tenure_band,count_star(),avg(churn_value)
0,Established,1856,0.255388
1,Loyal,3001,0.119294
2,New,2186,0.474382


## 7. The count of extra services, and the recording issue it exposes

This counts how many optional services each customer holds and shows the departure rate at each level.

This measure is the one most exposed to the recording problem found at Stage One. The six extra service fields hold three values rather than two: `Yes`, `No`, and `No internet service`. The third value does not mean a sale was turned down. It means the service was never available, because the customer holds no internet subscription at all.

In [15]:
con.execute(open("../sql/03_features.sql").read())
con.execute("SELECT addon_count, COUNT(*), AVG(churn_value) FROM vw_customer_features GROUP BY 1 ORDER BY 1").df()

,addon_count,count_star(),avg(churn_value)
0,0,2219,0.214060
1,1,966,0.457557
2,2,1033,0.358180
3,3,1118,0.273703
4,4,852,0.223005
5,5,571,0.124343
6,6,284,0.052817


The group holding no extra services needs a second look, because it contains two different kinds of customer: those who hold internet and took no extras, and those who hold no internet and could not have taken any.

The query below splits that group using the `has_internet` flag, so the two can be compared separately rather than averaged together.

In [16]:
con.execute("""
    SELECT has_internet, addon_count, COUNT(*) AS customers, AVG(churn_value) AS churn_rate
    FROM vw_customer_features
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()

,has_internet,addon_count,customers,churn_rate
0,0,0,1526,0.074050
1,1,0,693,0.522367
2,1,1,966,0.457557
3,1,2,1033,0.358180
4,1,3,1118,0.273703
5,1,4,852,0.223005
6,1,5,571,0.124343
7,1,6,284,0.052817


## 8. Working out customer value

This runs `04_clv.sql`, which creates `vw_customer_clv`, then shows the smallest, average and largest customer value within each contract type.

The `CLTV` field supplied with the dataset was rejected at Stage One, so customer value is worked out here from scratch and deliberately never checked against that field. The calculation rests on three assumptions, none of which are in the data: a profit margin of 30 percent, an expected remaining time set by contract type at twelve, twenty-four or thirty-six months, and no adjustment for the timing of payments. All three are stated in full in `04_clv.sql`, next to the calculation that uses them.

In [17]:
con.execute(open("../sql/04_clv.sql").read())
con.execute("""
    SELECT contract, COUNT(*) AS customers,
           ROUND(MIN(clv),2) AS min_clv,
           ROUND(AVG(clv),2) AS avg_clv,
           ROUND(MAX(clv),2) AS max_clv
    FROM vw_customer_clv GROUP BY 1 ORDER BY 3
""").df()

,contract,customers,min_clv,avg_clv,max_clv
0,Month-to-month,3875,67.50,239.03,422.82
1,One year,1473,131.40,468.35,853.92
2,Two year,1695,198.72,656.32,1282.50


## 9. Grouping customers by value

This runs `04_clv.sql` again and then summarises `vw_customer_value_segments`, showing the number of customers, average value, departure rate and total value in each value group.

`NTILE(10)` splits the customer base into ten equally sized groups ranked by customer value, with group 1 holding the highest-value customers. Groups are used rather than amounts in rand because the decision being supported is about relative standing, which customers rank highest, rather than whether a particular amount has been reached.

In [18]:
con.execute(open("../sql/04_clv.sql").read())
con.execute("""
    SELECT value_decile,
           COUNT(*) AS customers,
           ROUND(AVG(clv),2) AS avg_clv,
           ROUND(AVG(churn_value),3) AS churn_rate,
           ROUND(SUM(clv),2) AS total_value
    FROM vw_customer_value_segments
    GROUP BY 1 ORDER BY 1
""").df()

,value_decile,customers,avg_clv,churn_rate,total_value
0,1,705,1044.42,0.055,736317.18
1,2,705,723.70,0.138,510208.56
2,3,705,486.96,0.125,343307.16
3,4,704,354.67,0.517,249687.36
4,5,704,311.12,0.487,219027.96
5,6,704,276.15,0.411,194406.30
6,7,704,241.55,0.259,170054.46
7,8,704,199.72,0.188,140600.52
8,9,704,153.37,0.233,107973.54
9,10,704,80.99,0.243,57018.60


---

Stage Two ends here. The findings, including the departure pattern across tenure bands, the effect of the recording issue on the extra services count, and the identification of the value groups holding most of the value at risk, are set out in `md/02_schema_build_writeup.md`.

**Next stage:** build the model in `03_modelling.ipynb`, a logistic regression and a gradient boosting model, with one taken forward.